In [ ]:
import os
import tensorflow as tf
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 unused import

import matplotlib.pyplot as plt
import numpy as np
from scipy.special import erf,erfc,gamma
from tqdm import tqdm

np.random.seed(123)
tf.set_random_seed(123)
Modes = tf.estimator.ModeKeys

In [41]:
def trainer(opts,data,visdata_train,visdata_test,epoch,nr_samples,shuffle):
    #model
    model = HVAE(opts)
    batch_size = opts['batch_size']
    #training loop
    for ep in range(epoch):
        if shuffle:
            np.random.shuffle(data)
        for iter in tqdm(range(nr_samples // batch_size)):
            #obtain a mini-batch
            batch = data[iter*batch_size:(iter+1)*batch_size,:]
            loss, mloss, rloss, kloss, mu, sigma= model.run_single_step(batch)
 
        print('[Epoch {}] Recon Loss: {}, MMD: {}'.format(ep,loss,mloss))
        print('Mean: {}, "Var": {}'.format(mu,np.exp(0.5*sigma)))
        if opts['dataset'] == 'bd':
            subset_len = 1
            #draw_means(opts,visdata_train,model,treedepth=opts['treedepth'],
            #                 b=2,c=opts['c'],d=opts['input_dim'],subset_len=5)
            draw_latent_tree(opts,visdata_train,model,treedepth=opts['treedepth'],
                             b=2,c=opts['c'],d=opts['input_dim'],subset_len=1,plot_means=False)
            #draw_latent_tree(opts,visdata_test,model,treedepth=opts['treedepth'],
            #                 b=2,c=opts['c'],d=opts['input_dim'],subset_len=1,plot_means=False)

            
            z_test = model.transformer_mu(np.reshape(visdata_test[0:subset_len],[-1,opts['input_dim']]))
            data_u_tree = visdata_train[0:subset_len]
            data_v_tree = np.reshape(z_test,[subset_len,opts['b']**(opts['treedepth']+1),opts['c'],opts['dim_z']])
            avg_dist_tree = np_compute_avg_distortion_tree(data_u_tree, data_v_tree, b=opts['b'], treedepth=opts['treedepth'], c=opts['c'])

            data_u = np.reshape(visdata_test[0:subset_len],[-1,opts['input_dim']])[0:20]
            data_v = z_test[0:20]
            avg_dist_all = np_compute_avg_distortion_all(data_u, data_v)
            print('Average distortion: Tree: {}, All, {}'.format(avg_dist_tree, avg_dist_all))
        else:
            z_test = model.transformer_mu(np.reshape(visdata_test,[-1,opts['input_dim']]))
            data_u = np.reshape(visdata_test,[-1,opts['input_dim']])
            data_v = z_test
            avg_dist_all = np_compute_avg_distortion_all(data_u, data_v)
            print('Average distortion: All, {}'.format(avg_dist_all))
            
    print('Done!')
    return model

In [42]:
def load_wordnet_data(name):
    embeddings = np.load('./data/'+name+'.npy')
    np.random.shuffle(embeddings)
    N = len(embeddings)
    split = np.int(np.floor(0.8*N))
    embeddings_train = embeddings[:split]
    embeddings_test = embeddings[split:]
    return embeddings_train, embeddings_test

In [45]:
tf.reset_default_graph()
np.random.seed(4)
tf.set_random_seed(4)
opts = {}
opts['dataset'] = 'nnlm'
opts['input_dim'] = 50
if opts['dataset'] == 'nnlm':
    opts['input_dim'] = 128
opts['nw_type'] = 'mlp'
opts['lr_adam'] = 1e-3
opts['lr_rsgd'] = 1e-3
opts['batch_size'] = 64
opts['nn_act_fn'] = tf.nn.relu
opts['beta_mmd'] = 10.0
opts['sigma_mmd'] = 1.0 # Laplacian kernel width
opts['mmd_q'] = 1
opts['varregnorm'] = 2 # L1 or L2
opts['lambda_mmd'] = 1.0 # variance regularizer weight to avoid variance collapse
opts['beta_kld'] = 10.0
opts['dim_z'] = 2
opts['hyp_c'] = 1.0
opts['use_expmap'] = False
opts['hyp_var'] = False
opts['scalar_dispersion'] = False
opts['init_std'] = 0.01
opts['init_bias'] = 0.0
opts['prior_var'] = 1.2
opts['reparam_eucl'] = True
opts['b'] = 2
opts['treedepth'] = 6
opts['loss'] = 'l2' # l2 or bce loss
opts['recon_weight'] = 1.0
opts['c'] = 5
opts['dec_mlr'] = False

print(opts)
if opts['dataset'] == 'bd':
    X_train, X_train_flat, X_test = generate_tree_dataset(n_train=10,n_test=10,scale=1/opts['c'],
                                        treedepth=opts['treedepth'],b=opts['b'],c=opts['c'],d=opts['input_dim'],offset=2,
                                                     strict_norms=True,center=True)
    model = trainer(opts, data=X_train_flat, 
                visdata_train=X_train, visdata_test=X_test, epoch=1000, nr_samples=len(X_train_flat), shuffle=False) 
else:
    X_train, X_test = load_wordnet_data('nnlm')
    model = trainer(opts, data=X_train, 
                visdata_train=X_train, visdata_test=X_test, epoch=1000, nr_samples=len(X_train), shuffle=False) 


{'lr_rsgd': 0.001, 'prior_var': 1.2, 'lr_adam': 0.001, 'use_expmap': False, 'c': 5, 'scalar_dispersion': False, 'init_std': 0.01, 'dec_mlr': False, 'hyp_c': 1.0, 'hyp_var': False, 'nn_act_fn': <function relu at 0x7f909b160400>, 'sigma_mmd': 1.0, 'varregnorm': 2, 'treedepth': 6, 'lambda_mmd': 1.0, 'b': 2, 'beta_kld': 10.0, 'loss': 'l2', 'batch_size': 64, 'dataset': 'nnlm', 'mmd_q': 1, 'beta_mmd': 10.0, 'input_dim': 128, 'recon_weight': 1.0, 'nw_type': 'mlp', 'reparam_eucl': True, 'dim_z': 2, 'init_bias': 0.0}
Tensor("sub_157:0", shape=(?, 2), dtype=float32)
Tensor("gradients/hyp_ff_layer_mu/ExpandDims_grad/Reshape:0", shape=(2,), dtype=float32) <tf.Variable 'hyp_ff_layer_mu/hypbias:0' shape=(2,) dtype=float32_ref>
INFO:tensorflow:Summary name var/enc_fc1/kernel:0 is illegal; using var/enc_fc1/kernel_0 instead.
INFO:tensorflow:Summary name var/enc_fc1/bias:0 is illegal; using var/enc_fc1/bias_0 instead.
INFO:tensorflow:Summary name var/enc_fc2/kernel:0 is illegal; using var/enc_fc2/kerne

100%|██████████| 10/10 [00:02<00:00,  4.38it/s]


[Epoch 0] Recon Loss: 41.766380310058594, MMD: 0.04529289901256561
Mean: [0.22890414 0.31169638], "Var": [1.0136223 1.0083752]


100%|██████████| 10/10 [00:00<00:00, 131.16it/s]

Average distortion: All, 0.7874085903167725
[Epoch 1] Recon Loss: 33.86988067626953, MMD: 0.03560236841440201
Mean: [0.18189219 0.36226907], "Var": [0.9885516  0.98891395]


...


100%|██████████| 10/10 [00:00<00:00, 134.83it/s]

Average distortion: All, 7.4188079833984375
[Epoch 116] Recon Loss: 20.853492736816406, MMD: 0.05988013371825218
Mean: [ 0.32560092 -0.06399561], "Var": [0.99844253 0.9990135 ]



100%|██████████| 10/10 [00:00<00:00, 137.42it/s]

Average distortion: All, 7.762713432312012
[Epoch 117] Recon Loss: 20.969186782836914, MMD: 0.04793648421764374
Mean: [ 0.3384619  -0.03763125], "Var": [1.0026671 1.0022308]


KeyboardInterrupt: 

In [ ]:
draw_means(opts,X_train,model,treedepth=opts['treedepth'],
                             b=2,c=opts['c'],d=opts['input_dim'],subset_len=10)
draw_latent_tree(opts,X_train,model,treedepth=opts['treedepth'],
                             b=2,c=opts['c'],d=opts['input_dim'],subset_len=10,plot_means=False)